# 1-Year Cumulative Incidence Heatmap by Cancer Type (Figure 2E)

Heatmap of 1-year cumulative incidence (%) for 6 irAE types across top 10 cancer types + Other. Column-normalized coloring.

Also includes: per-(cancer type, toxicity) one-vs-rest Cox proportional-hazards models, exported to CSV for QC (see bottom of notebook).

In [ ]:
%matplotlib inline
import re, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 450,
})
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.backends.backend_pdf import PdfPages
from lifelines import KaplanMeierFitter, CoxPHFitter

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r"\d+", str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None

TOXICITY_COLUMNS = ["pneumonitis", "adrenal_insufficiency", "liver_toxicity",
                    "colitis", "hyperthyroidism", "hypothyroidism"]
TOXICITY_DISPLAY = {
    "pneumonitis": "Pneumonitis", "adrenal_insufficiency": "Adrenal Insufficiency",
    "liver_toxicity": "Liver Toxicity", "colitis": "Colitis",
    "hyperthyroidism": "Hyperthyroidism", "hypothyroidism": "Hypothyroidism",
}

In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
TOX_TABLE_DIR = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'main'))
os.makedirs(RESULTS_DIR, exist_ok=True)

BACKBONE_PATH = os.path.join(TOX_TABLE_DIR, 'llm84k_pneumonitis_grade0_20260630.csv')
covars = pd.read_csv(BACKBONE_PATH, low_memory=False)
batch_df = pd.read_csv(os.path.join(DATA_DIR, 'llm_calls_batch_level_84k.csv'), encoding="latin-1", low_memory=False)

# Standardize MRNs
covars["mrn"] = covars["mrn"].apply(standardize_mrn)
batch_df["mrn"] = batch_df["mrn"].apply(standardize_mrn)
covars = covars[covars["mrn"].notna()].copy()
batch_df = batch_df[batch_df["mrn"].notna()].copy()

# Parse dates
covars["lot_start"] = pd.to_datetime(covars["lot_start"], errors="coerce")
covars["lot"] = pd.to_numeric(covars["lot"], errors="coerce")
covars["censor_days"] = pd.to_numeric(covars["t_cutoff_lot"], errors="coerce")
covars = covars.sort_values(["mrn", "lot"])

batch_df["window_start"] = pd.to_datetime(batch_df["window_start"], errors="coerce")
batch_df["window_end"] = pd.to_datetime(batch_df["window_end"], errors="coerce")
batch_df = batch_df.dropna(subset=["window_start", "window_end"])

# First LOT per patient (for cancer_type -- already on the backbone, no separate merge needed)
patient_covars = covars.groupby("mrn").first().reset_index()
print(f"Patients with first-LOT covariates: {len(patient_covars):,}")

In [ ]:
covars_valid = covars[covars["lot_start"].notna()].copy()
line1 = (covars_valid.sort_values(["mrn", "lot"])
         .groupby("mrn").first().reset_index()[["mrn", "lot_start", "censor_days"]]
         .rename(columns={"lot_start": "line1_start"}))
line1 = line1[np.isfinite(line1["censor_days"]) & (line1["censor_days"] > 0)].copy()
print(f"Line 1 patients with valid censoring: {len(line1):,}")

In [ ]:
# Assign cancer types: top 10 + Other (exclude Multiple Cancer Type Patient)
ct_col_df = patient_covars[["mrn", "cancer_type"]].drop_duplicates("mrn")
line1_ct = line1.merge(ct_col_df, on="mrn", how="left")
line1_ct = line1_ct[line1_ct["cancer_type"].notna() & (line1_ct["cancer_type"] != "")].copy()
line1_ct = line1_ct[line1_ct["cancer_type"] != "Multiple Cancer Type Patient"]

ct_counts = line1_ct["cancer_type"].value_counts()
top10 = ct_counts.head(10).index.tolist()
line1_ct.loc[~line1_ct["cancer_type"].isin(top10), "cancer_type"] = "Other"
cancer_order = [c for c in top10 if c in line1_ct["cancer_type"].unique()] + ["Other"]

print("Cancer types:")
for ct in cancer_order:
    n = (line1_ct["cancer_type"] == ct).sum()
    print(f"  {ct}: n={n:,}")

In [ ]:
# Merge batch with line1
batch_merged = batch_df.merge(line1, on="mrn", how="inner")
batch_merged["days_from_start"] = (batch_merged["window_start"] - batch_merged["line1_start"]).dt.days
batch_merged = batch_merged[
    (batch_merged["days_from_start"] >= 0) &
    (batch_merged["days_from_start"] <= batch_merged["censor_days"])
].copy()
print(f"Batch records after merge & filtering: {len(batch_merged):,}")

In [ ]:
# Compute 1-year CI matrix: cancer type x toxicity
T_MONTHS = 12.0
ae_list = TOXICITY_COLUMNS
display_labels = [TOXICITY_DISPLAY[t] for t in ae_list]

matrix = np.zeros((len(cancer_order), len(ae_list)))
row_labels = []

for i, ct in enumerate(cancer_order):
    ct_mrns = set(line1_ct.loc[line1_ct["cancer_type"] == ct, "mrn"])
    n_ct = len(ct_mrns)
    row_labels.append(f"{ct} (n={n_ct:,})")
    ct_l1 = line1[line1["mrn"].isin(ct_mrns)].copy()
    ct_batch = batch_merged[batch_merged["mrn"].isin(ct_mrns)].copy()
    for j, tox in enumerate(ae_list):
        if tox not in ct_batch.columns or n_ct < 10:
            continue
        ae_rec = ct_batch[ct_batch[tox] == 1].copy()
        ae_rec = ae_rec[ae_rec["days_from_start"] >= 0]
        first_ae = ae_rec.groupby("mrn")["days_from_start"].min().reset_index().rename(
            columns={"days_from_start": "time"})
        first_ae["event"] = 1
        surv = ct_l1[["mrn", "censor_days"]].merge(first_ae, on="mrn", how="left")
        surv["event"] = surv["event"].fillna(0).astype(int)
        surv.loc[surv["event"] == 0, "time"] = surv.loc[surv["event"] == 0, "censor_days"]
        surv.loc[(surv["event"] == 1) & (surv["time"] > surv["censor_days"]), "event"] = 0
        surv.loc[surv["event"] == 0, "time"] = surv["censor_days"]
        surv = surv[surv["time"] > 0].copy()
        if len(surv) < 10:
            continue
        surv["time_months"] = surv["time"] / 30.44
        kmf = KaplanMeierFitter()
        kmf.fit(surv["time_months"], surv["event"])
        matrix[i, j] = (1 - kmf.survival_function_at_times(T_MONTHS).values[0]) * 100
    print(f"  {ct} done")

print("CI matrix computed.")

In [ ]:
# Column-normalize for coloring
col_mins = matrix.min(axis=0, keepdims=True)
col_maxs = matrix.max(axis=0, keepdims=True)
col_range = np.where(col_maxs == col_mins, 1, col_maxs - col_mins)
normed = (matrix - col_mins) / col_range

# Plot
fig, ax = plt.subplots(figsize=(2.50, 2.22))
im = ax.imshow(normed, aspect="auto", cmap="YlOrRd", interpolation="nearest", vmin=0, vmax=1)

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        color = "white" if normed[i, j] > 0.55 else "black"
        ax.text(j, i, f"{val:.1f}%", ha="center", va="center", fontsize=5,
                color=color, fontweight="bold")

ax.set_xticks(range(len(display_labels)))
ax.set_xticklabels(display_labels, fontsize=6, rotation=40, ha="right", rotation_mode="anchor")
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=6)
ax.tick_params(axis="both", length=0)

cbar = plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02, aspect=25)
cbar.set_label("Relative 1-year CI" + chr(10) + "(column-normalized)", fontsize=7)
cbar.set_ticks([0, 0.5, 1.0])
cbar.set_ticklabels(["Low", "Mid", "High"])
cbar.ax.tick_params(labelsize=6)

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
with PdfPages(os.path.join(RESULTS_DIR, 'Heatmap_CI_By_Cancer_Type_2E.pdf')) as pdf:
    pdf.savefig(fig, dpi=450, bbox_inches="tight", pad_inches=0.05)
plt.close(fig)
print("Saved: ../results/main/Heatmap_CI_By_Cancer_Type_2E.pdf")

---
## Cox proportional-hazards models — one-vs-rest, per cancer type × toxicity

For QC purposes. First-line-treatment cohort (same `line1` cohort as the heatmap above), excluding patients with `sex == "Unknown"`.

Cancer type is dummy-coded so that each of the 11 cancer types (top 10 + Other, matching the heatmap's row categories) gets its own HR/CI/p-value "vs. all others" — the same one-vs-rest design used in the treatment-category analysis (Figure 2E), just with cancer type as the exposure instead of treatment category. This gives **11 cancer types × 6 toxicities = 66 models**, not one model per toxicity.

Each model adjusts for: `has_pd1_flag` and `has_ctla4_flag` (combined the same way as panel 2E — each ORs together its two raw source columns), `contains_chemo`, `contains_hormone`, `contains_biologic`, `contains_targeted`, then age (`age_at_lot_start`), then sex. Since no single `contains_*` flag defines a cancer type, none of these adjustment covariates are excluded from any model (unlike 2E, where the flag defining the current exposure was left out).

Cancer type (the exposure) is protected the same way treatment exposure was protected in 2E: it is never dropped for events-per-parameter reasons. If a specific cancer-type/toxicity combination is truly zero-event (unfittable), that entire model is skipped and marked accordingly, rather than dropping the variable the model exists to test. All other adjustment covariates go through the same two-stage process as 2E: a zero-cell check (binary covariates only, `age` exempt) removes structurally unfittable covariates first, then an events-per-parameter rule (~10 events/parameter) trims further from the lowest-priority end (sex, then age, then the contains-flags) if still underpowered. A light L2 penalty (`penalizer=0.1`) is applied to every fitted model as a safety net for near-separation.

In [ ]:
def _flag_on(val):
    return val in [1, True, "1", "True"]

# Build patient-level model dataframe: one row per Line-1 patient
model_df = patient_covars.drop(columns=["censor_days"], errors="ignore").merge(line1, on="mrn", how="inner").copy()

# Exclude Unknown sex
model_df = model_df[model_df["sex"] != "Unknown"].copy()
print(f"Model cohort after excluding Unknown sex: {len(model_df):,} patients")

# Combined immuno flags -- same logic as panel 2E
model_df["has_ctla4_flag"] = (model_df["contains_ctla4_immuno"].apply(_flag_on) |
                                model_df["contains_ctla4"].apply(_flag_on)).astype(int)
model_df["has_pd1_flag"] = (model_df["contains_non_ctla4_immuno"].apply(_flag_on) |
                              model_df["contains_pd1"].apply(_flag_on)).astype(int)

for col in ["contains_chemo", "contains_hormone", "contains_biologic", "contains_targeted"]:
    model_df[col] = model_df[col].apply(_flag_on).astype(int) if col in model_df.columns else 0

ADJUST_COLS = ["has_pd1_flag", "has_ctla4_flag",
               "contains_chemo", "contains_hormone", "contains_biologic", "contains_targeted"]

# Age (real column is age_at_lot_start) and sex
model_df["age"] = model_df["age_at_lot_start"]
model_df["sex_male"] = (model_df["sex"] == "MALE").astype(int)

# Cancer type: same top-10 + Other categories as the heatmap
model_df = model_df.merge(ct_col_df, on="mrn", how="left", suffixes=("", "_dup"))
model_df = model_df[model_df["cancer_type"].notna() & (model_df["cancer_type"] != "")].copy()
model_df = model_df[model_df["cancer_type"] != "Multiple Cancer Type Patient"]
model_df.loc[~model_df["cancer_type"].isin(top10), "cancer_type"] = "Other"

print(f"Model cohort: {len(model_df):,} patients")
print(f"Cancer type categories: {cancer_order}")
print(f"sex_male distribution: {model_df['sex_male'].value_counts().to_dict()}")

In [ ]:
def get_survival_for_tox(mrn_frame, tox):
    """Full follow-up time/event (not landmarked), same censoring rules as elsewhere."""
    sub_b = batch_merged[batch_merged["mrn"].isin(mrn_frame["mrn"])].copy()
    ae_rec = sub_b[sub_b[tox] == 1].copy() if tox in sub_b.columns else pd.DataFrame()
    ae_rec = ae_rec[ae_rec["days_from_start"] >= 0] if len(ae_rec) > 0 else ae_rec
    if len(ae_rec) > 0:
        first_ae = ae_rec.groupby("mrn")["days_from_start"].min().reset_index().rename(
            columns={"days_from_start": "time"})
        first_ae["event"] = 1
    else:
        first_ae = pd.DataFrame(columns=["mrn", "time", "event"])
    surv = mrn_frame[["mrn", "censor_days"]].merge(first_ae, on="mrn", how="left")
    surv["event"] = surv["event"].fillna(0).astype(int)
    surv.loc[surv["event"] == 0, "time"] = surv.loc[surv["event"] == 0, "censor_days"]
    surv.loc[(surv["event"] == 1) & (surv["time"] > surv["censor_days"]), "event"] = 0
    surv.loc[surv["event"] == 0, "time"] = surv["censor_days"]
    surv = surv[surv["time"] > 0].copy()
    surv["time_months"] = surv["time"] / 30.44
    return surv[["mrn", "time_months", "event"]]

def has_zero_cell(df, col, event_col="event"):
    """True if this BINARY covariate has 0 events in its 0-group or its 1-group.
    Not meaningful for continuous covariates -- caller should skip those."""
    if df[col].nunique() < 2:
        return True
    events_by_group = df.groupby(col)[event_col].sum()
    return (events_by_group == 0).any()

def slug(s):
    return (s.lower().replace(" ", "_").replace("(", "").replace(")", "")
            .replace("+", "plus").replace("-", "_"))

In [ ]:
MIN_EVENTS_PER_PARAM = 10
PENALIZER = 0.1

BINARY_COVARIATES = set(ADJUST_COLS + ["sex_male"])

all_results = []

for ct in cancer_order:
    for tox in TOXICITY_COLUMNS:
        surv = get_survival_for_tox(model_df, tox)
        cox_df = model_df.merge(surv, on="mrn", how="inner")
        cox_df["exposure"] = (cox_df["cancer_type"] == ct).astype(int)
        cox_df = cox_df.dropna(subset=["age", "sex_male", "time_months", "event"])

        n_total = len(cox_df)
        n_events = int(cox_df["event"].sum())
        model_id = f"{slug(TOXICITY_DISPLAY[tox])}_{slug(ct)}"

        # Guard: is the exposure itself unfittable (this cancer type has 0 events for this AE)?
        if has_zero_cell(cox_df, "exposure"):
            all_results.append(pd.DataFrame([{
                "model_id": model_id, "cancer_type": ct, "toxicity": TOXICITY_DISPLAY[tox],
                "covariate": None, "n_total": n_total, "n_events": n_events,
                "covariates_dropped_zero_cell": "", "covariates_dropped_low_epv": "",
                "model_status": "skipped: exposure has zero-event cell"
            }]))
            print(f"  {ct} / {TOXICITY_DISPLAY[tox]}: SKIPPED (exposure zero-cell), n={n_total:,}, events={n_events:,}")
            continue

        priority_covariates = ADJUST_COLS + ["age"] + ["sex_male"]

        # Step 1: drop any BINARY adjustment covariate with a zero-event cell.
        # Exposure (cancer type) is never dropped here -- only skipped above if unfittable.
        zero_cell_dropped = []
        covariates_to_use = []
        for cov in priority_covariates:
            if cov in BINARY_COVARIATES and has_zero_cell(cox_df, cov):
                zero_cell_dropped.append(cov)
            else:
                covariates_to_use.append(cov)

        # Step 2: events-per-parameter rule, drop from lowest-priority end.
        # +1 accounts for exposure, which is always in the model and never dropped here.
        epv_dropped = []
        while covariates_to_use and (n_events / (len(covariates_to_use) + 2)) < MIN_EVENTS_PER_PARAM:
            d = covariates_to_use.pop()
            epv_dropped.append(d)

        model_cols = ["time_months", "event", "exposure"] + covariates_to_use
        fit_df = cox_df[model_cols].dropna()

        try:
            cph = CoxPHFitter(penalizer=PENALIZER)
            cph.fit(fit_df, duration_col="time_months", event_col="event")
            summary = cph.summary.reset_index().rename(columns={"index": "covariate"})
            summary["hr"] = np.exp(summary["coef"])
            summary["hr_lower_95"] = np.exp(summary["coef lower 95%"])
            summary["hr_upper_95"] = np.exp(summary["coef upper 95%"])
            summary["model_id"] = model_id
            summary["cancer_type"] = ct
            summary["toxicity"] = TOXICITY_DISPLAY[tox]
            summary["n_total"] = n_total
            summary["n_events"] = n_events
            summary["covariates_dropped_zero_cell"] = ", ".join(zero_cell_dropped) if zero_cell_dropped else ""
            summary["covariates_dropped_low_epv"] = ", ".join(epv_dropped) if epv_dropped else ""
            summary["model_status"] = "fit"
        except Exception as e:
            summary = pd.DataFrame([{
                "model_id": model_id, "cancer_type": ct, "toxicity": TOXICITY_DISPLAY[tox],
                "covariate": None, "n_total": n_total, "n_events": n_events,
                "covariates_dropped_zero_cell": ", ".join(zero_cell_dropped) if zero_cell_dropped else "",
                "covariates_dropped_low_epv": ", ".join(epv_dropped) if epv_dropped else "",
                "model_status": f"failed: {e}"
            }])
        all_results.append(summary)
        print(f"  {ct} / {TOXICITY_DISPLAY[tox]}: n={n_total:,}, events={n_events:,}, "
              f"zero_cell_dropped={zero_cell_dropped if zero_cell_dropped else 'none'}, "
              f"epv_dropped={epv_dropped if epv_dropped else 'none'}")

cox_results_df = pd.concat(all_results, ignore_index=True)

### Export — full model results (every covariate, every model) for QC.

Long format: HR / 95% CI / p-value for every term (cancer-type exposure + all adjustment covariates) across all up-to-66 models, plus N, event count, which covariates were dropped and why, fit status, and `model_id` for quick lookup (e.g. `pneumonitis_non_small_cell_lung_cancer`).

In [ ]:
cols_order = ["model_id", "cancer_type", "toxicity", "covariate", "hr", "hr_lower_95", "hr_upper_95",
              "p", "n_total", "n_events", "covariates_dropped_zero_cell", "covariates_dropped_low_epv",
              "model_status"]
cox_results_df = cox_results_df[[c for c in cols_order if c in cox_results_df.columns]]
cox_results_df.to_csv(os.path.join(RESULTS_DIR, 'Heatmap_CI_By_Cancer_Type_2E_cox_models.csv'), index=False)
cox_results_df